# Aurora Alpha process management

Local owner/operator notebook for the serving-only Aurora Alpha process. **Run All is read-only by default.** It checks the local package manifest, listener, health endpoint and widget frame; it does not create a chat session, install packages, call a model, alter a release, touch customer data, or perform housekeeping.

This notebook controls only this Aurora process. It does not manage Ollama, Neo, Oracle, or other services. The configured listener is restricted to `127.0.0.1`. To start/stop, set `ALLOW_PROCESS_ACTIONS = True` in the configuration cell, run only the relevant action cell, and enter its exact confirmation. Stop is PID-specific and refuses a process that does not match this app, host, port and release.

The current package/release values are local staging references. Review and update them after deliberately building a new package. The site API key must be supplied as `AURORA_SITE_API_KEY_LOCAL` in the notebook process environment; never put it in this notebook or the site JSON. Do not point this notebook at a public listener or enter real customer details into the local demo.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import subprocess
import sys
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

def find_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'scripts' / 'package_runtime.py').is_file() and (candidate / 'web_agent.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook from this repository or one of its subdirectories.')

ROOT = find_root()
PACKAGE_DIR = ROOT.joinpath('data', 'local', 'distribution', 'aurora-chat-fix-20261006').resolve()
STATE_DIR = Path(os.environ['LOCALAPPDATA']).joinpath('Aurora', 'staging', 'state-1134bcd1').resolve()
SITE_DIR = PACKAGE_DIR.joinpath('config', 'sites')
LOG_FILE = Path(os.environ['LOCALAPPDATA']).joinpath('Aurora', 'staging', 'logs', 'aurora-alpha-managed.log')
EXPECTED_RELEASE_ID = '1134bcd195cba93072f0f7190219b0a572d6c7dc899ac27cbbbaf11c5eae72f6'
SITE_ID = 'local'
HOST = '127.0.0.1'
PORT = 8011
ALLOW_PROCESS_ACTIONS = False

print('Workspace found:', ROOT)
print('Serving package:', PACKAGE_DIR)
print('Expected release:', EXPECTED_RELEASE_ID)
print('Loopback endpoint:', f'http://{HOST}:{PORT}')
print('Process actions enabled:', ALLOW_PROCESS_ACTIONS)


In [ ]:
def checked_package():
    distribution = (ROOT / 'data' / 'local' / 'distribution').resolve()
    package = PACKAGE_DIR.resolve()
    if not package.is_relative_to(distribution) or package == distribution:
        raise ValueError('Package must be a named folder under data/local/distribution.')
    manifest = json.loads((package / 'package_manifest.json').read_text(encoding='utf-8'))
    if manifest.get('default_profile') != 'serving-only' or manifest.get('release_id') != EXPECTED_RELEASE_ID:
        raise ValueError('Package profile/release differs from the explicitly expected Alpha package.')
    pointer = json.loads((package / 'releases' / 'active.json').read_text(encoding='utf-8'))
    if pointer.get('release_id') != EXPECTED_RELEASE_ID:
        raise ValueError('Package release pointer does not match the expected release.')
    for relative, digest in manifest.get('files', {}).items():
        path = (package / Path(*PurePosixPath(relative).parts)).resolve()
        if not path.is_relative_to(package) or not path.is_file():
            raise ValueError('Package manifest contains an absent or out-of-package file.')
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise ValueError('Package checksum mismatch: ' + relative)
    config_path = SITE_DIR / f'{SITE_ID}.json'
    config = json.loads(config_path.read_text(encoding='utf-8'))
    if (config.get('site_id') != SITE_ID or config.get('api_key')
            or config.get('allowed_origins') != [f'http://{HOST}:{PORT}']):
        raise ValueError('Demo site must have no file-based API key and allow only this exact loopback origin.')
    if not STATE_DIR.is_dir():
        raise FileNotFoundError('Existing state directory not found; refusing to create or redirect state.')
    return manifest

def powershell_json(script):
    result = subprocess.run(
        ['powershell.exe', '-NoProfile', '-NonInteractive', '-Command', script],
        capture_output=True, text=True, check=True,
    )
    text = result.stdout.strip()
    return json.loads(text) if text else []

def listeners():
    rows = powershell_json(
        f"$items = @(Get-NetTCPConnection -State Listen -LocalPort {PORT} "
        "-ErrorAction SilentlyContinue | Select-Object LocalAddress,LocalPort,OwningProcess); "
        'ConvertTo-Json -InputObject $items -Compress'
    )
    return rows if isinstance(rows, list) else [rows]

def process_command_line(pid):
    if type(pid) is not int or pid <= 0:
        raise ValueError('Invalid listener PID.')
    item = powershell_json(
        f"$p = Get-CimInstance Win32_Process -Filter 'ProcessId = {pid}'; "
        'if ($p) { $p | Select-Object ProcessId,CommandLine | ConvertTo-Json -Compress }'
    )
    if isinstance(item, list):
        return item[0] if item else None
    return item or None

def get_json(url):
    with urlopen(url, timeout=3) as response:
        return response.status, json.loads(response.read().decode('utf-8'))

def status_snapshot():
    manifest = checked_package()
    rows = listeners()
    addresses = sorted({str(row.get('LocalAddress', '')) for row in rows})
    pids = sorted({int(row['OwningProcess']) for row in rows if row.get('OwningProcess') is not None})
    report = {
        'package_verified': True,
        'expected_release': manifest['release_id'],
        'listener_addresses': addresses,
        'listener_pids': pids,
        'loopback_only': bool(rows) and addresses == [HOST],
        'serving_only_ready': False,
        'release_matches': False,
        'widget_frame_status': None,
        'widget_script_status': None,
    }
    if len(pids) == 1 and report['loopback_only']:
        command = process_command_line(pids[0])
        text = str((command or {}).get('CommandLine', '')).casefold()
        report['aurora_process_matches'] = all(token in text for token in (
            'uvicorn web_agent:app', f'--host {HOST}', f'--port {PORT}', '--workers 1',
        ))
        try:
            _, ready = get_json(f'http://{HOST}:{PORT}/health/ready')
            report['serving_only_ready'] = ready.get('status') == 'ready' and ready.get('serving_only') is True
            report['release_matches'] = ready.get('release_id') == EXPECTED_RELEASE_ID
            query = urlencode({'site_id': SITE_ID, 'parent_origin': f'http://{HOST}:{PORT}'})
            with urlopen(f'http://{HOST}:{PORT}/widget?{query}', timeout=3) as response:
                report['widget_frame_status'] = response.status
            with urlopen(f'http://{HOST}:{PORT}/widget.js', timeout=3) as response:
                report['widget_script_status'] = response.status
        except (HTTPError, URLError, TimeoutError, OSError, ValueError) as exc:
            report['health_error'] = type(exc).__name__
    else:
        report['aurora_process_matches'] = False
    return report

snapshot = status_snapshot()
for key, value in snapshot.items():
    print(f'{key}: {value}')


## Start Aurora Alpha (explicit action)

This cell does nothing while `ALLOW_PROCESS_ACTIONS` is false. If enabled, it verifies package hashes, release pointer, site config, state directory and that port 8011 has no listener; it then requires the exact release-specific confirmation. The child is detached from the notebook kernel. Output is appended to the private local management log. A startup failure is not automatically killed; inspect the PID and log first.


In [ ]:
if not ALLOW_PROCESS_ACTIONS:
    print('No action: explicitly enable ALLOW_PROCESS_ACTIONS before running this cell.')
else:
    manifest = checked_package()
    if listeners():
        raise RuntimeError('Port 8011 is occupied. Inspect the listener; do not start a second server.')
    phrase = input(f"Type START AURORA {EXPECTED_RELEASE_ID[:16]} to launch the verified package: ")
    if phrase != f'START AURORA {EXPECTED_RELEASE_ID[:16]}':
        print('Cancelled; no process was started.')
    else:
        LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
        if not os.environ.get('AURORA_SITE_API_KEY_LOCAL', '').strip():
            raise RuntimeError('Required site API key is absent from the notebook environment; do not add it to the notebook or site config.')
        inherited = ('PATH', 'SYSTEMROOT', 'WINDIR', 'TEMP', 'TMP', 'LOCALAPPDATA', 'APPDATA', 'USERPROFILE', 'AURORA_SITE_API_KEY_LOCAL')
        runtime_env = {key: os.environ[key] for key in inherited if key in os.environ}
        runtime_env.update({
            'AURORA_ENV': 'production',
            'AURORA_SERVING_ONLY': 'true',
            'AURORA_RELEASE_DIR': str(PACKAGE_DIR / 'releases'),
            'AURORA_STATE_DIR': str(STATE_DIR),
            'AURORA_SITES_DIR': str(SITE_DIR),
            'AGENT_USE_LLM': 'false',
            'USE_HYBRID_RANKING': 'false',
            'AURORA_SESSION_BACKEND': 'sqlite',
            'AURORA_RATE_LIMIT_BACKEND': 'sqlite',
        })
        flags = getattr(subprocess, 'DETACHED_PROCESS', 0x00000008) | getattr(subprocess, 'CREATE_NEW_PROCESS_GROUP', 0x00000200)
        with LOG_FILE.open('ab') as log:
            process = subprocess.Popen(
                [sys.executable, '-m', 'uvicorn', 'web_agent:app', '--app-dir', str(PACKAGE_DIR),
                 '--host', HOST, '--port', str(PORT), '--workers', '1'],
                cwd=str(PACKAGE_DIR), env=runtime_env, stdin=subprocess.DEVNULL,
                stdout=log, stderr=subprocess.STDOUT, creationflags=flags,
            )
        print('Started PID:', process.pid)
        print('Private log:', LOG_FILE)
        for _ in range(30):
            time.sleep(1)
            try:
                report = status_snapshot()
                if report['serving_only_ready'] and report['release_matches'] and report['loopback_only']:
                    print('Ready: serving-only release and loopback checks passed.')
                    break
            except (OSError, ValueError):
                continue
        else:
            print('Not verified ready. Do not start another process; inspect the exact PID and private log.')


## Stop Aurora Alpha (explicit action)

This cell does nothing while actions are disabled. If enabled, it will stop only the sole listener on `127.0.0.1:8011` after confirming its Uvicorn app, worker count, ready release and serving-only mode. It requires the exact PID displayed by the read-only status cell. It never kills by process name. To restart, stop and then run the start cell; verify readiness again.


In [ ]:
if not ALLOW_PROCESS_ACTIONS:
    print('No action: explicitly enable ALLOW_PROCESS_ACTIONS before running this cell.')
else:
    report = status_snapshot()
    pids = report['listener_pids']
    safe_to_stop = (
        len(pids) == 1 and report.get('loopback_only') and report.get('aurora_process_matches')
        and report.get('serving_only_ready') and report.get('release_matches')
    )
    if not safe_to_stop:
        raise RuntimeError('Refusing to stop: listener/process/release did not pass all safety checks.')
    pid = pids[0]
    phrase = input(f'Type STOP AURORA {pid} to stop this exact process: ')
    if phrase != f'STOP AURORA {pid}':
        print('Cancelled; no process was stopped.')
    else:
        subprocess.run(
            ['powershell.exe', '-NoProfile', '-NonInteractive', '-Command', f'Stop-Process -Id {pid} -ErrorAction Stop'],
            check=True, capture_output=True, text=True,
        )
        time.sleep(1)
        remaining = [row for row in listeners() if int(row.get('OwningProcess', -1)) == pid]
        if remaining:
            raise RuntimeError('Stop was issued but the verified PID still has a listener.')
        print('Stopped verified Aurora PID:', pid)


## Operator notes

- Keep this notebook and the repository local. It contains no customer messages or credentials.
- Do not display raw process command lines, environment values, site-config contents, logs, or SQLite rows; the status code intentionally prints only booleans, IDs, addresses and PIDs.
- The package check verifies each manifest-listed file against its SHA-256 and requires the configured exact release. A changed package requires a deliberate update of `PACKAGE_DIR` and `EXPECTED_RELEASE_ID` here.
- This is a process manager, not a release builder, backup/restore tool, data editor, retention controller, public deployment installer or production approval.
- For release/backup/retention procedures and incident guidance, see `docs/AURORA_MANAGEMENT.md` and `docs/LOCAL_DEPLOYMENT.md`.
